In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
sample_text = """
Retrieval-Augmented Generation (RAG) is a technique that bridges the gap between 
a large language model's general knowledge and your specific, private data. 
By using local models like Llama 3.1 and local embeddings, you ensure that 
sensitive information never leaves your machine. The first step in any RAG 
pipeline is data ingestion, which involves loading documents and splitting 
them into chunks. Proper chunking is critical because if chunks are too small, 
they lose semantic meaning. If they are too large, they might exceed the 
model's context window or dilute the specific answer you are looking for.
""" * 10 

with open("local_rag_guide.txt", "w") as file:
    file.write(sample_text)

from langchain_community.document_loaders import TextLoader
loader = TextLoader("local_rag_guide.txt")
docs = loader.load()

print(f"Successfully loaded {len(docs)} document(s).")
print(f"Total character count: {len(docs[0].page_content)}")

small_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=10)
medium_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
large_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

small_chunks = small_splitter.split_documents(docs)
medium_chunks = medium_splitter.split_documents(docs)
large_chunks = large_splitter.split_documents(docs)

from langchain_community.embeddings import OllamaEmbeddings

ollama_embeddings = OllamaEmbeddings(model="nomic-embed-text")


Right now, our chunks and our embeddings only exist in your computer's temporary RAM. If you close VS Code, they vanish. If you want to chat with a 100-page PDF, you do not want to wait for the embedding model to process all 100 pages every single time you start your app.

But a normal database (like SQL or an Excel spreadsheet) is terrible at comparing 768-dimensional mathematical vectors. We need a specialized Vector Database. We are going to use ChromaDB, which is open-source, runs entirely locally, and saves the data right into a folder in your project.

In [ ]:
from langchain_community.vectorstores import Chroma
import os

# 2. Define a folder name where the database will be saved
persist_directory = "./chroma_db"

# 3. Initialize the Vector Database
print("Creating vector database. This might take a moment...")

vector_db = Chroma.from_documents(
    documents=medium_chunks, # Using the medium chunks we made in Lesson 1
    embedding=ollama_embeddings, # Using the fast Nomic embeddings
    # If we didn't use persist_directory, ChromaDB would run purely "in-memory." 
    # That means every single time you wanted to chat with your PDF, you would have to wait for your computer to re-read the PDF, 
    # re-chunk it, and re-embed all the text all over again.
    persist_directory=persist_directory 
)

print(f"Success! Database created and saved to {persist_directory}")

Creating vector database. This might take a moment...
Success! Database created and saved to ./chroma_db


**So essentially what we've done is we've chunked down the text, and then used a certain embedding model to embed it - the chunking affects how it judges the similarity** <br>
Embedding effects:
* Too Large (The Kitchen Sink Smoothie): Imagine a massive chunk of text that talks about a company's Refund Policy, its CEO's History, and its Product Specs. When the embedding model processes that chunk, it mathematically blends all three topics together.
If a user searches "What is the refund policy?", the system might not find that chunk. Why? Because the math representing the CEO and the Product Specs "pulled" the vector away from the pure concept of refunds. The meaning got diluted.
* Too Small (The Single Ingredient): Imagine a tiny chunk that just says, "It takes 30 days." When embedded, the math just represents the concept of "time passing." If the user searches "How long do refunds take?", the system won't match it, because the tiny chunk is completely missing the concept of a "refund."
* Just Right (The Perfect Chunk): If you chunk it perfectly so one chunk is exactly one cohesive paragraph about the Refund Policy, its mathematical vector will intensely point exactly to the concept of refunds. When the user asks about refunds, the system will instantly recognize it as a near-perfect mathematical match.